# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahmedali3ff/Flyrank-internship-/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [116]:
import os

print(os.listdir("/content"))

['.config', 'Flyrank-internship-', 'sample_data']


In [117]:
!git clone https://github.com/Ahmedali3ff/Flyrank-internship-.github

Cloning into 'Flyrank-internship-.github'...
fatal: could not read Username for 'https://github.com': No such device or address


In [118]:
import os

DATA_PATH = "/content/Flyrank-internship-/data/raw/content_refresh_anonymized.csv"

print("File exists:", os.path.exists(DATA_PATH))
print(DATA_PATH)

File exists: True
/content/Flyrank-internship-/data/raw/content_refresh_anonymized.csv


In [119]:
!git clone https://github.com/Ahmedali3ff/Flyrank-internship-.git /content/Flyrank-internship-

fatal: destination path '/content/Flyrank-internship-' already exists and is not an empty directory.


In [120]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
        precision_score,
            recall_score,
                f1_score,
                    roc_auc_score,
                        confusion_matrix
                        )

DATA_PATH = "/content/Flyrank-internship-/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)

                        # Target
df["is_declining_label"] = (
                            df["trend_direction"].astype(str).str.lower() == "down"
                            ).astype(int)

print("Target definition: trend_direction == 'down'")
print("Declining rate:", round(df["is_declining_label"].mean(), 4))

Dataset shape: (30000, 44)
Target definition: trend_direction == 'down'
Declining rate: 0.5421


In [121]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [122]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [123]:
feature_cols = [
      "search_volume",
          "competition",
              "cpc",
                  "word_count",
                      "char_count",
                          "impressions_90d",
                              "clicks_90d",
                                  "pageviews_90d",
                                      "sessions_90d",
                                          "users_90d",
                                              "engaged_sessions_90d",
                                                  "ai_sessions_90d",
                                                      "scroll_events_90d",
                                                          "days_with_impressions",
                                                              "days_with_sessions",
                                                                  "impressions_last_30d",
                                                                      "clicks_last_30d",
                                                                          "sessions_last_30d",
                                                                              "impressions_prev_30d",
                                                                                  "clicks_prev_30d",
                                                                                      "sessions_prev_30d",
                                                                                          "content_age_days",
                                                                                              "days_since_last_update",
                                                                                                  "ctr",
                                                                                                      "avg_position",
                                                                                                          "engagement_rate",
                                                                                                              "scroll_rate",
                                                                                                                  "ai_traffic_pct"
                                                                                                                  ]

X = df[feature_cols].copy()
y = df["is_declining_label"].copy()

X_train, X_test, y_train, y_test = train_test_split(
                                                                                                                      X,
                                                                                                                          y,
                                                                                                                              test_size=0.20,
                                                                                                                                  random_state=42,
                                                                                                                                      stratify=y
                                                                                                                                      )

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Train declining rate:", round(y_train.mean(), 4))
print("Test declining rate:", round(y_test.mean(), 4))


Training rows: 24000
Test rows: 6000
Train declining rate: 0.5421
Test declining rate: 0.542


In [124]:
# Evaluate Logistic Regression
logreg_accuracy = accuracy_score(y_test, y_pred)
logreg_precision = precision_score(y_test, y_pred)
logreg_recall = recall_score(y_test, y_pred)
logreg_f1 = f1_score(y_test, y_pred)
logreg_roc_auc = roc_auc_score(y_test, y_prob)

print("LOGISTIC REGRESSION RESULTS")
print("---------------------------")
print(f"Accuracy : {logreg_accuracy:.4f}")
print(f"Precision: {logreg_precision:.4f}")
print(f"Recall   : {logreg_recall:.4f}")
print(f"F1-Score : {logreg_f1:.4f}")
print(f"ROC-AUC  : {logreg_roc_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

LOGISTIC REGRESSION RESULTS
---------------------------
Accuracy : 0.8218
Precision: 0.8429
Recall   : 0.8250
F1-Score : 0.8339
ROC-AUC  : 0.9124

Confusion Matrix:
[[2248  500]
 [ 569 2683]]


In [125]:
# Recreate Week-4 baseline prediction

# Staleness score
staleness_score_map = {
    "(-inf, 30.0]": 0,
        "(30.0, 90.0]": 1,
            "(90.0, 180.0]": 2,
                "(180.0, 365.0]": 3,
                    "(365.0, inf]": 4
                    }

                    # Position score
position_score_map = {
                        "Top 3": 0,
                            "4-10": 1,
                                "11-20": 2,
                                    "21+": 3
                                    }

                                    # Create buckets
df["staleness_bucket"] = pd.cut(
                                        df["content_age_days"],
                                            bins=[-float("inf"), 30, 90, 180, 365, float("inf")]
                                            )

df["position_bucket"] = pd.cut(
                                                df["avg_position"],
                                                    bins=[-float("inf"), 3, 10, 20, float("inf")],
                                                        labels=["Top 3", "4-10", "11-20", "21+"]
                                                        )

                                                        # Convert buckets to scores
df["staleness_score"] = (
                                                            df["staleness_bucket"]
                                                                .astype(str)
                                                                    .map(staleness_score_map)
                                                                        .fillna(0)
                                                                        )
df["position_score"] = (
                                                                            df["position_bucket"]
                                                                                .astype(str)
                                                                                    .map(position_score_map)
                                                                                        .fillna(0)
                                                                                        )

                                                                                        # Baseline score
df["baseline_score"] = (
                                                                                            df["staleness_score"] + df["position_score"]
                                                                                            )

                                                                                            # Baseline prediction
df["baseline_prediction"] = (
                                                                                                df["baseline_score"] >= 4
                                                                                                ).astype(int)

print("Baseline predictions:", df["baseline_prediction"].sum())

Baseline predictions: 22138


In [126]:
# Model vs Week-4 Baseline

baseline_test = df.loc[X_test.index, "baseline_prediction"]

comparison = pd.DataFrame({
    "Model": [
            "Week-4 Baseline",
                    "Logistic Regression"
                        ],
                            "Accuracy": [
                                    accuracy_score(y_test, baseline_test),
                                            accuracy_score(y_test, y_pred)
                                                ],
                                                    "Precision": [
                                                            precision_score(y_test, baseline_test),
                                                                    precision_score(y_test, y_pred)
                                                                        ],
                                                                            "Recall": [
                                                                                    recall_score(y_test, baseline_test),
                                                                                            recall_score(y_test, y_pred)
                                                                                                ],
                                                                                                    "F1": [
                                                                                                            f1_score(y_test, baseline_test),
                                                                                                                    f1_score(y_test, y_pred)
                                                                                                                        ]
                                                                                                                        })
display(comparison)

,Model,Accuracy,Precision,Recall,F1
0,Week-4 Baseline,0.540333,0.555781,0.756765,0.640885
1,Logistic Regression,0.821833,0.842915,0.825031,0.833877


In [127]:
# Error Analysis

from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print("LOGISTIC REGRESSION ERROR ANALYSIS")
print("----------------------------------")
print("True Negatives :", cm[0, 0])
print("False Positives:", cm[0, 1])
print("False Negatives:", cm[1, 0])
print("True Positives :", cm[1, 1])

print("\nInterpretation:")
print(
    f"The model correctly identified {cm[1,1]} declining-content cases "
        f"and missed {cm[1,0]} declining cases."
        )


print(
            f"It incorrectly flagged {cm[0,1]} non-declining cases as declining."
            )

LOGISTIC REGRESSION ERROR ANALYSIS
----------------------------------
True Negatives : 2248
False Positives: 500
False Negatives: 569
True Positives : 2683

Interpretation:
The model correctly identified 2683 declining-content cases and missed 569 declining cases.
It incorrectly flagged 500 non-declining cases as declining.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [128]:
model = Pipeline([
      ("imputer", SimpleImputer(strategy="median")),
          ("scaler", StandardScaler()),
              ("classifier", LogisticRegression(
                      max_iter=1000,
                              random_state=42
                                  ))
                                  ])

model.fit(X_train, y_train)

model_pred = model.predict(X_test)
model_prob = model.predict_proba(X_test)[:, 1]

print("Logistic Regression trained successfully.")


Logistic Regression trained successfully.


In [129]:
# Feature Interpretation

# Get the trained Logistic Regression model
model = logreg_pipeline.named_steps["model"]

# Get the preprocessor
preprocessor = logreg_pipeline.named_steps["preprocessor"]

# Get feature names
feature_names = preprocessor.get_feature_names_out()

# Get Logistic Regression coefficients
coefficients = model.coef_[0]

# Create feature importance table
feature_importance = pd.DataFrame({
    "Feature": feature_names,
        "Coefficient": coefficients
        })

        # Absolute coefficient = strength of influence
feature_importance["Absolute_Coefficient"] = (
            feature_importance["Coefficient"].abs()
            )

            # Sort by strongest influence
feature_importance = feature_importance.sort_values(
                "Absolute_Coefficient",
                    ascending=False
                    )

print("TOP 15 FEATURES")
print("----------------")

display(feature_importance.head(15))

TOP 15 FEATURES
----------------


,Feature,Coefficient,Absolute_Coefficient
14,num__impressions_last_30d,-35.277107,35.277107
17,num__impressions_prev_30d,28.861864,28.861864
5,num__impressions_90d,1.464111,1.464111
7,num__sessions_90d,1.105403,1.105403
15,num__clicks_last_30d,-0.980841,0.980841
18,num__clicks_prev_30d,0.802855,0.802855
16,num__sessions_last_30d,-0.662379,0.662379
12,num__days_with_impressions,0.513166,0.513166
13,num__days_with_sessions,-0.472504,0.472504
11,num__scroll_events_90d,-0.376551,0.376551


In [130]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# Numeric features
numeric_features = [
    "search_volume",
        "competition",
            "cpc",
                "word_count",
                    "char_count",
                        "impressions_90d",
                            "clicks_90d",
                                "sessions_90d",
                                    "users_90d",
                                        "engaged_sessions_90d",
                                            "ai_sessions_90d",
                                                "scroll_events_90d",
                                                    "days_with_impressions",
                                                        "days_with_sessions",
                                                            "impressions_last_30d",
                                                                "clicks_last_30d",
                                                                    "sessions_last_30d",
                                                                        "impressions_prev_30d",
                                                                            "clicks_prev_30d",
                                                                                "sessions_prev_30d",
                                                                                    "content_age_days",
                                                                                        "days_since_last_update",
                                                                                            "ctr",
                                                                                                "avg_position",
                                                                                                    "engagement_rate",
                                                                                                        "scroll_rate",
                                                                                                            "ai_traffic_pct"
                                                                                                            ]

                                                                                                            # Keep only columns that actually exist
numeric_features = [c for c in numeric_features if c in X_train.columns]

preprocessor = ColumnTransformer(
                                                                                                                transformers=[
                                                                                                                        (
                                                                                                                                    "num",
                                                                                                                                                Pipeline([
                                                                                                                                                                ("imputer", SimpleImputer(strategy="median")),
                                                                                                                                                                                ("scaler", StandardScaler())
                                                                                                                                                                                            ]),
                                                                                                                                                                                                        numeric_features
                                                                                                                                                                                                                )
                                                                                                                                                                                                                    ],
                                                                                                                                                                                                                        remainder="drop"
                                                                                                                                                                                                                        )

logreg_pipeline = Pipeline([
                                                                                                                                                                                                                            ("preprocessor", preprocessor),
                                                                                                                                                                                                                                ("model", LogisticRegression(
                                                                                                                                                                                                                                        max_iter=1000,
                                                                                                                                                                                                                                                random_state=42
                                                                                                                                                                                                                                                    ))
                                                                                                                                                                                                                                                    ])

                                                                                                                                                                                                                                                    # Train
logreg_pipeline.fit(X_train, y_train)

print("Logistic Regression trained successfully.")

Logistic Regression trained successfully.


In [131]:
# Predictions
y_pred = logreg_pipeline.predict(X_test)
y_prob = logreg_pipeline.predict_proba(X_test)[:, 1]

print("Predictions generated successfully.")
print("Number of predictions:", len(y_pred))

Predictions generated successfully.
Number of predictions: 6000


In [132]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [133]:
# Recreate Week-4 baseline

eval_df = df.loc[X_test.index].copy()

eval_df["staleness_bucket"] = pd.cut(
    eval_df["content_age_days"],
        bins=[-float("inf"), 30, 90, 180, 365, float("inf")]
        )

eval_df["position_bucket"] = pd.cut(
            eval_df["avg_position"],
                bins=[-float("inf"), 3, 10, 20, float("inf")],
                    labels=["Top 3", "4-10", "11-20", "21+"]
                    )

staleness_score_map = {
                        "(-inf, 30.0]": 0,
                            "(30.0, 90.0]": 1,
                                "(90.0, 180.0]": 2,
                                    "(180.0, 365.0]": 3,
                                        "(365.0, inf]": 4
                                        }

osition_score_map = {
                                            "Top 3": 0,
                                                "4-10": 1,
                                                    "11-20": 2,
                                                        "21+": 3
                                                        }

eval_df["staleness_score"] = (
                                                            eval_df["staleness_bucket"]
                                                                .astype(str)
                                                                    .map(staleness_score_map)
                                                                        .fillna(0)
                                                                        )

position_score_map = {
      "Top 3": 0,
          "4-10": 1,
              "11-20": 2,
                  "21+": 3
                  }

eval_df["position_score"] = (
                                                                            eval_df["position_bucket"]
                                                                                .astype(str)
.map(position_score_map)
                                                                                        .fillna(0)
                                                                                        )

eval_df["baseline_score"] = (
                                                                                            eval_df["staleness_score"] +
                                                                                                eval_df["position_score"]
                                                                                                )

eval_df["baseline_prediction"] = (
                                                                                                    eval_df["baseline_score"] >= 4
                                                                                                    ).astype(int)

baseline_pred = eval_df["baseline_prediction"].values

print("Baseline predictions:", baseline_pred.sum())

Baseline predictions: 4428


In [134]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [135]:
# Errors and Interpretation

print("ERRORS AND INTERPRETATION")
print("------------------------")

print(f"False Positives: {cm[0,1]}")
print(f"False Negatives: {cm[1,0]}")
print(f"True Positives: {cm[1,1]}")
print(f"True Negatives: {cm[0,0]}")

print("\nInterpretation:")
print(
    "The Logistic Regression model substantially outperformed "
        "the Week-4 rule-based baseline."
        )

print(
            f"F1 improved from {comparison.loc[0, 'F1']:.3f} "
                f"to {comparison.loc[1, 'F1']:.3f}."
                )

print(
                    "The strongest model signals were recent and previous-period "
                        "impressions, followed by longer-term traffic and engagement features."
                        )

print(
                            "The coefficient signs describe how the model uses each standardized "
                                "feature when estimating the probability of decline; they should not "
                                    "be interpreted as causal effects."
                                    )

ERRORS AND INTERPRETATION
------------------------
False Positives: 500
False Negatives: 569
True Positives: 2683
True Negatives: 2248

Interpretation:
The Logistic Regression model substantially outperformed the Week-4 rule-based baseline.
F1 improved from 0.641 to 0.834.
The strongest model signals were recent and previous-period impressions, followed by longer-term traffic and engagement features.
The coefficient signs describe how the model uses each standardized feature when estimating the probability of decline; they should not be interpreted as causal effects.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [136]:
# Self-check

print("SELF-CHECK")
print("----------")
print("✓ Same dataset used as Week-4 baseline")
print("✓ 80/20 train-test split")
print("✓ No test data used for training")
print("✓ Baseline and Logistic Regression evaluated on same test set")
print("✓ Model substantially improves over Week-4 baseline")
print("✓ Errors and feature coefficients inspected")

SELF-CHECK
----------
✓ Same dataset used as Week-4 baseline
✓ 80/20 train-test split
✓ No test data used for training
✓ Baseline and Logistic Regression evaluated on same test set
✓ Model substantially improves over Week-4 baseline
✓ Errors and feature coefficients inspected
